In [ ]:
# 기본 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path

# 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

# 시각화 스타일
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 한글 폰트 설정 (선택사항)
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 경로
DATA_PATH = Path('data')

# 사용 가능한 CSV 파일 확인
csv_files = list(DATA_PATH.glob('*.csv'))

print(f"발견된 CSV 파일: {len(csv_files)}개\n")
for file in sorted(csv_files):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"  • {file.name:<30} ({size_mb:>8.2f} MB)")

### 각 테이블 기본 정보 확인

In [ ]:
# 각 파일의 기본 정보를 딕셔너리로 저장
table_info = {}

for csv_file in sorted(csv_files):
    table_name = csv_file.stem  # 파일명 (확장자 제외)
    
    # 첫 5줄만 읽어서 구조 파악
    df_sample = pd.read_csv(csv_file, nrows=5)
    
    table_info[table_name] = {
        'columns': df_sample.columns.tolist(),
        'dtypes': df_sample.dtypes.to_dict(),
        'shape': df_sample.shape,
        'sample': df_sample
    }
    
    print(f"\n{'='*60}")
    print(f"테이블: {table_name}")
    print(f"{'='*60}")
    print(f"컬럼 수: {len(df_sample.columns)}")
    print(f"컬럼 목록: {', '.join(df_sample.columns)}")
    print(f"\n샘플 데이터:")
    display(df_sample.head(5))


**Hosp/Patients Table**

In [ ]:
# Patients 테이블 전체 로드
print("Patients 테이블 분석\n")

df_patients = pd.read_csv(DATA_PATH / 'patients.csv')

print(f"총 환자 수: {len(df_patients):,}명")
print(f"컬럼: {df_patients.columns.tolist()}\n")

# 기본 통계
print("=" * 50)
print("기본 통계:")
print("=" * 50)
display(df_patients.describe(include='all'))

# 결측치 확인
print("\n" + "=" * 50)
print("결측치 현황:")
print("=" * 50)
missing = df_patients.isnull().sum()
missing_pct = (missing / len(df_patients) * 100).round(2)
missing_df = pd.DataFrame({
    '결측 수': missing,
    '결측률(%)': missing_pct
})
display(missing_df[missing_df['결측 수'] > 0])

# 성별 분포
print("\n" + "=" * 50)
print("성별 분포:")
print("=" * 50)
print(df_patients['gender'].value_counts())

# 나이 분포 (anchor_age 기준)
print("\n" + "=" * 50)
print("나이 분포:")
print("=" * 50)
print(df_patients['anchor_age'].describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('patients 테이블 환자 인구통계학적 특성', fontsize=16, fontweight='bold')

# 성별 분포
ax1 = axes[0, 0]
sns.countplot(data=df_patients, x='gender', palette=['#3498db', '#e74c3c'], 
              edgecolor='black', linewidth=2, ax=ax1)
ax1.set_title('성별 분포', fontsize=12, fontweight='bold')
ax1.set_xlabel('성별')
ax1.set_ylabel('환자 수')
ax1.set_xticklabels(['여성', '남성'])

for i, p in enumerate(ax1.patches):
    height = p.get_height()
    percentage = height / len(df_patients) * 100
    ax1.text(p.get_x() + p.get_width()/2., height + 1000,
             f'{int(height):,}명\n({percentage:.1f}%)',
             ha='center', va='bottom', fontweight='bold')

# 나이 분포
ax2 = axes[0, 1]
sns.histplot(data=df_patients, x='anchor_age', kde=True, 
             color='#2ecc71', edgecolor='black', bins=30, ax=ax2, alpha=0.7)
ax2.set_title('나이 분포', fontsize=12, fontweight='bold')
ax2.set_xlabel('나이 (세)')
ax2.set_ylabel('빈도')

mean_age = df_patients['anchor_age'].mean()
median_age = df_patients['anchor_age'].median()
ax2.axvline(mean_age, color='red', linestyle='--', linewidth=2.5, 
            label=f'평균: {mean_age:.1f}세')
ax2.axvline(median_age, color='blue', linestyle='--', linewidth=2.5, 
            label=f'중앙값: {median_age:.1f}세')
ax2.legend()

# 성별별 나이 분포
ax3 = axes[1, 0]
sns.boxplot(data=df_patients, x='gender', y='anchor_age', 
            palette=['#3498db', '#e74c3c'], ax=ax3,
            showmeans=True,
            meanprops=dict(marker='D', markerfacecolor='yellow', markersize=10,
                          markeredgecolor='black', markeredgewidth=1.5))
ax3.set_title('성별별 나이 분포', fontsize=12, fontweight='bold')
ax3.set_xlabel('성별')
ax3.set_ylabel('나이 (세)')
ax3.set_xticklabels(['여성', '남성'])
ax3.grid(axis='y', alpha=0.3, linestyle='--')

for i, gender in enumerate(['F', 'M']):
    mean_val = df_patients[df_patients['gender'] == gender]['anchor_age'].mean()
    ax3.text(i, mean_val, f'평균: {mean_val:.1f}세', 
             ha='center', va='bottom', fontsize=9,
             bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.8))

# 사망 여부
ax4 = axes[1, 1]
death_status = df_patients['dod'].notna()
death_counts = death_status.value_counts()
colors_pie = ['#95a5a6', '#34495e']
wedges, texts, autotexts = ax4.pie(death_counts.values, 
                                     labels=['생존', '사망'],
                                     autopct='%1.1f%%',
                                     colors=colors_pie,
                                     startangle=90,
                                     explode=(0, 0.1),
                                     shadow=True)
ax4.set_title('사망 여부', fontsize=12, fontweight='bold')
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)

plt.tight_layout()
plt.show()

print(f"\n사망 환자 수: {death_status.sum():,}명 ({death_status.sum()/len(df_patients)*100:.2f}%)")

**Host/admissions table**

In [ ]:
# Admissions 테이블 로드
print("Admissions 테이블 분석\n")

df_admissions = pd.read_csv(DATA_PATH / 'admissions.csv')

print(f"총 입원 기록: {len(df_admissions):,}건")
print(f"고유 환자 수: {df_admissions['subject_id'].nunique():,}명")
print(f"평균 입원 횟수: {len(df_admissions) / df_admissions['subject_id'].nunique():.2f}회/인\n")

# 기본 정보
print("=" * 50)
print("컬럼 정보:")
print("=" * 50)
print(df_admissions.info())

# 입원 유형 분포
print("\n" + "=" * 50)
print("입원 유형 (Admission Type):")
print("=" * 50)
print(df_admissions['admission_type'].value_counts())

# 입원 경로
print("\n" + "=" * 50)
print("입원 경로 (Admission Location):")
print("=" * 50)
print(df_admissions['admission_location'].value_counts().head(10))

# 퇴원 경로
print("\n" + "=" * 50)
print("퇴원 경로 (Discharge Location):")
print("=" * 50)
print(df_admissions['discharge_location'].value_counts().head(10))

# 원내 사망
print("\n" + "=" * 50)
print("원내 사망 (Hospital Expire Flag):")
print("=" * 50)
expire_counts = df_admissions['hospital_expire_flag'].value_counts()
print(f"생존 퇴원: {expire_counts.get(0, 0):,}건")
print(f"원내 사망: {expire_counts.get(1, 0):,}건")
print(f"사망률: {expire_counts.get(1, 0) / len(df_admissions) * 100:.2f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Admissions 테이블 시각화', fontsize=16, fontweight='bold')

# 입원 유형 분포
ax1 = axes[0, 0]
admission_type = df_admissions['admission_type'].value_counts()
ax1.barh(admission_type.index, admission_type.values, color='#3498db')
ax1.set_title('입원 유형 분포', fontsize=12, fontweight='bold')
ax1.set_xlabel('건수')
for i, (idx, val) in enumerate(admission_type.items()):
    ax1.text(val + 100, i, f'{val:,}', va='center', fontweight='bold')

# 입원 경로 Top 10
ax2 = axes[0, 1]
admission_loc = df_admissions['admission_location'].value_counts().head(10)
ax2.barh(range(len(admission_loc)), admission_loc.values, color='#2ecc71')
ax2.set_yticks(range(len(admission_loc)))
ax2.set_yticklabels(admission_loc.index, fontsize=9)
ax2.set_title('입원 경로 Top 10', fontsize=12, fontweight='bold')
ax2.set_xlabel('건수')
ax2.invert_yaxis()

# 퇴원 경로 Top 10
ax3 = axes[1, 0]
discharge_loc = df_admissions['discharge_location'].value_counts().head(10)
ax3.barh(range(len(discharge_loc)), discharge_loc.values, color='#e74c3c')
ax3.set_yticks(range(len(discharge_loc)))
ax3.set_yticklabels(discharge_loc.index, fontsize=9)
ax3.set_title('퇴원 경로 Top 10', fontsize=12, fontweight='bold')
ax3.set_xlabel('건수')
ax3.invert_yaxis()

# 보험 유형 분포
ax4 = axes[1, 1]
insurance = df_admissions['insurance'].value_counts()
colors_insurance = plt.cm.Set3(range(len(insurance)))
wedges, texts, autotexts = ax4.pie(insurance.values,
                                     labels=insurance.index,
                                     autopct='%1.1f%%',
                                     colors=colors_insurance,
                                     startangle=90)
ax4.set_title('보험 유형 분포', fontsize=12, fontweight='bold')
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

**Host/diagnose_ICD table**

ICD 코드란?
ICD (International Classification of Diseases)
- WHO에서 제정한 국제 질병 분류 체계
- 모든 질병을 코드화하여 표준화된 진단 기록 가능
- 현재 ICD-9과 ICD-10이 주로 사용됨

심부전 관련 코드
- ICD-10 : 150.x
- ICD-9 : 428.x

In [ ]:
print("Diagnoses_ICD 테이블 분석 (심부전 중심)\n")

df_diagnoses = pd.read_csv(DATA_PATH / 'd_icd_diagnoses.csv')

print(f"총 진단 코드 수: {len(df_diagnoses):,}개")
print(f"ICD-9 코드: {(df_diagnoses['icd_version'] == 9).sum():,}개")
print(f"ICD-10 코드: {(df_diagnoses['icd_version'] == 10).sum():,}개\n")

# 심부전 관련 코드 찾기
print("=" * 80)
print("심부전 관련 ICD 코드:")
print("=" * 80)

# ICD-10: I50.x
hf_icd10 = df_diagnoses[
    (df_diagnoses['icd_code'].str.startswith('I50', na=False)) &
    (df_diagnoses['icd_version'] == 10)
]

print("\n[ICD-10 코드]")
display(hf_icd10[['icd_code', 'long_title']].sort_values('icd_code'))

# ICD-9: 428.x
hf_icd9 = df_diagnoses[
    (df_diagnoses['icd_code'].str.startswith('428', na=False)) &
    (df_diagnoses['icd_version'] == 9)
]

print("\n[ICD-9 코드]")
display(hf_icd9[['icd_code', 'long_title']].sort_values('icd_code'))

# 심부전 코드 리스트 저장 (나중에 사용)
hf_codes = pd.concat([hf_icd10['icd_code'], hf_icd9['icd_code']]).tolist()
print(f"\n✓ 총 {len(hf_codes)}개의 심부전 관련 ICD 코드 확인됨")

In [ ]:
print("실제 진단 기록에서 심부전 환자 찾기\n")

# 실제 진단 데이터 로드
df_diagnoses_records = pd.read_csv(DATA_PATH / 'diagnoses_icd.csv')

print(f"총 진단 기록: {len(df_diagnoses_records):,}건")
print(f"고유 입원 건수: {df_diagnoses_records['hadm_id'].nunique():,}건")
print(f"고유 환자 수: {df_diagnoses_records['subject_id'].nunique():,}명\n")

# 심부전 진단 필터링
hf_diagnoses = df_diagnoses_records[
    df_diagnoses_records['icd_code'].isin(hf_codes)
]

print("=" * 80)
print("심부전 진단 통계:")
print("=" * 80)
print(f"심부전 진단 기록: {len(hf_diagnoses):,}건")
print(f"심부전 입원 건수: {hf_diagnoses['hadm_id'].nunique():,}건")
print(f"심부전 환자 수: {hf_diagnoses['subject_id'].nunique():,}명")
print(f"전체 환자 중 비율: {hf_diagnoses['subject_id'].nunique() / df_patients['subject_id'].nunique() * 100:.2f}%")

# 가장 흔한 심부전 코드
print("\n" + "=" * 80)
print("가장 흔한 심부전 진단 코드 Top 10:")
print("=" * 80)
top_hf_codes = hf_diagnoses['icd_code'].value_counts().head(10)

# 코드 설명과 함께 출력
for code, count in top_hf_codes.items():
    # 코드 설명 찾기
    code_info = df_diagnoses[df_diagnoses['icd_code'] == code]
    if not code_info.empty:
        title = code_info.iloc[0]['long_title']
        version = code_info.iloc[0]['icd_version']
        print(f"{code:10} (ICD-{version}) | {count:>6,}건 | {title}")
    else:
        print(f"{code:10} | {count:>6,}건 | (설명 없음)")

# 샘플 확인
print("\n" + "=" * 80)
print("심부전 진단 샘플 데이터:")
print("=" * 80)
display(hf_diagnoses.head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('심부전 진단 분포', fontsize=16, fontweight='bold')

# 1. Top 10 심부전 코드
ax1 = axes[0]
top_codes = hf_diagnoses['icd_code'].value_counts().head(10)
bars = ax1.barh(range(len(top_codes)), top_codes.values, color='#e74c3c', edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(top_codes)))
ax1.set_yticklabels([f"{code}" for code in top_codes.index], fontsize=10, family='monospace')
ax1.set_title('심부전 진단 코드 Top 10', fontsize=13, fontweight='bold')
ax1.set_xlabel('진단 건수', fontsize=11)
ax1.invert_yaxis()
ax1.grid(axis='x', alpha=0.3, linestyle='--')

# 값 표시
for i, (idx, val) in enumerate(top_codes.items()):
    ax1.text(val + 20, i, f'{val:,}', va='center', fontweight='bold', fontsize=10)

# 2. ICD 버전별 분포
ax2 = axes[1]

# ICD 버전별 집계 (diagnoses_icd.csv의 icd_version 사용)
version_counts = hf_diagnoses['icd_version'].value_counts().sort_index()

colors = ['#3498db', '#2ecc71']
labels = [f'ICD-{int(v)}' for v in version_counts.index]

wedges, texts, autotexts = ax2.pie(version_counts.values,
                                     labels=labels,
                                     autopct='%1.1f%%',
                                     colors=colors[:len(version_counts)],
                                     startangle=90,
                                     explode=[0.05] * len(version_counts),
                                     shadow=True)
ax2.set_title('ICD 버전별 심부전 진단 분포', fontsize=13, fontweight='bold')

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')

for text in texts:
    text.set_fontsize(11)
    text.set_fontweight('bold')

plt.tight_layout()
plt.show()

# 추가 통계
print("\n" + "=" * 50)
print("ICD 버전별 상세 통계")
print("=" * 50)
for version in version_counts.index:
    count = version_counts[version]
    pct = count / len(hf_diagnoses) * 100
    print(f"ICD-{int(version)}: {count:,}건 ({pct:.1f}%)")

In [ ]:
print("심부전 유형별 상세 분석\n")

# 심부전 진단에 설명 추가
hf_diagnoses_detailed = hf_diagnoses.merge(
    df_diagnoses[['icd_code', 'icd_version', 'long_title']],
    on=['icd_code', 'icd_version'],
    how='left'
)

# 유형별 분류 (키워드 기반)
def classify_hf_type(title):
    if pd.isna(title):
        return '기타'
    title_lower = title.lower()
    
    if 'acute' in title_lower:
        return '급성'
    elif 'chronic' in title_lower:
        return '만성'
    elif 'systolic' in title_lower:
        return '수축기'
    elif 'diastolic' in title_lower:
        return '이완기'
    elif 'congestive' in title_lower:
        return '울혈성'
    elif 'left' in title_lower:
        return '좌심실'
    elif 'right' in title_lower:
        return '우심실'
    else:
        return '기타/미분류'

hf_diagnoses_detailed['hf_type'] = hf_diagnoses_detailed['long_title'].apply(classify_hf_type)

# 유형별 통계
print("=" * 80)
print("심부전 유형별 분포:")
print("=" * 80)
type_counts = hf_diagnoses_detailed['hf_type'].value_counts()
for hf_type, count in type_counts.items():
    pct = count / len(hf_diagnoses_detailed) * 100
    print(f"{hf_type:15} : {count:>6,}건 ({pct:>5.1f}%)")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('심부전 유형별 분석', fontsize=16, fontweight='bold')

# 1. 유형별 막대 그래프
ax1 = axes[0]
type_counts_sorted = type_counts.sort_values(ascending=True)
colors_bar = plt.cm.Set3(range(len(type_counts_sorted)))
bars = ax1.barh(range(len(type_counts_sorted)), type_counts_sorted.values, 
                color=colors_bar, edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(type_counts_sorted)))
ax1.set_yticklabels(type_counts_sorted.index, fontsize=11)
ax1.set_xlabel('진단 건수', fontsize=11)
ax1.set_title('심부전 유형별 분포', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3, linestyle='--')

for i, val in enumerate(type_counts_sorted.values):
    ax1.text(val + 20, i, f'{val:,}', va='center', fontweight='bold')

# 2. 파이 차트
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(type_counts.values,
                                     labels=type_counts.index,
                                     autopct='%1.1f%%',
                                     colors=plt.cm.Set3(range(len(type_counts))),
                                     startangle=90)
ax2.set_title('심부전 유형별 비율', fontsize=13, fontweight='bold')
for text in texts:
    text.set_fontsize(10)
for autotext in autotexts:
    autotext.set_fontsize(9)
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

In [ ]:
print("심부전 유형별 상세 분석\n")

# 심부전 진단에 설명 추가
hf_diagnoses_detailed = hf_diagnoses.merge(
    df_diagnoses[['icd_code', 'icd_version', 'long_title']],
    on=['icd_code', 'icd_version'],
    how='left'
)

# 개선된 분류 함수
def classify_hf_comprehensive(row):
    code = str(row['icd_code'])
    title = str(row['long_title']).lower() if pd.notna(row['long_title']) else ''
    version = row['icd_version']
    
    # 1. 시간 경과 분류 (우선순위 높음)
    temporal = None
    if 'acute' in title and 'chronic' in title:
        temporal = '급성 악화 (Acute on Chronic)'
    elif 'acute' in title:
        temporal = '급성 (Acute)'
    elif 'chronic' in title:
        temporal = '만성 (Chronic)'
    
    # 2. 기능적 분류
    functional = None
    if 'systolic' in title and 'diastolic' in title:
        functional = '복합형 (Combined)'
    elif 'systolic' in title:
        functional = '수축기 (Systolic)'
    elif 'diastolic' in title:
        functional = '이완기 (Diastolic)'
    
    # 3. 해부학적 분류
    anatomical = None
    if 'biventricular' in title:
        anatomical = '양심실 (Biventricular)'
    elif 'left' in title:
        anatomical = '좌심실 (Left)'
    elif 'right' in title:
        anatomical = '우심실 (Right)'
    
    # 4. 특수 유형
    special = None
    if 'congestive' in title:
        special = '울혈성 (Congestive)'
    elif 'high output' in title:
        special = '고박출량 (High-output)'
    elif 'end stage' in title or 'end-stage' in title:
        special = '말기 (End-stage)'
    
    # 5. ICD 코드 기반 분류 (보조)
    if version == 9:
        if code == '4280':
            special = '울혈성 (Congestive)'
        elif code == '4281':
            anatomical = '좌심실 (Left)'
    elif version == 10:
        if code.startswith('I501'):
            anatomical = '좌심실 (Left)'
        elif code.startswith('I502'):
            functional = '수축기 (Systolic)'
        elif code.startswith('I503'):
            functional = '이완기 (Diastolic)'
        elif code.startswith('I504'):
            functional = '복합형 (Combined)'
        elif code.startswith('I5081'):
            anatomical = '우심실 (Right)'
        elif code.startswith('I5082'):
            anatomical = '양심실 (Biventricular)'
        elif code.startswith('I5084'):
            special = '말기 (End-stage)'
    
    # 우선순위에 따라 반환
    if temporal:
        return temporal
    elif functional:
        return functional
    elif anatomical:
        return anatomical
    elif special:
        return special
    else:
        return '미분류 (Unspecified)'

# 분류 적용
hf_diagnoses_detailed['hf_type'] = hf_diagnoses_detailed.apply(
    classify_hf_comprehensive, axis=1
)

# 다차원 분류 (모든 특성 추출)
def extract_all_features(row):
    """각 진단의 모든 특성을 추출"""
    code = str(row['icd_code'])
    title = str(row['long_title']).lower() if pd.notna(row['long_title']) else ''
    
    features = {
        'temporal': [],
        'functional': [],
        'anatomical': [],
        'special': []
    }
    
    # Temporal
    if 'acute' in title and 'chronic' in title:
        features['temporal'].append('Acute on Chronic')
    elif 'acute' in title:
        features['temporal'].append('Acute')
    elif 'chronic' in title:
        features['temporal'].append('Chronic')
    
    # Functional
    if 'systolic' in title:
        features['functional'].append('Systolic')
    if 'diastolic' in title:
        features['functional'].append('Diastolic')
    
    # Anatomical
    if 'left' in title:
        features['anatomical'].append('Left')
    if 'right' in title:
        features['anatomical'].append('Right')
    if 'biventricular' in title:
        features['anatomical'].append('Biventricular')
    
    # Special
    if 'congestive' in title:
        features['special'].append('Congestive')
    if 'end stage' in title or 'end-stage' in title:
        features['special'].append('End-stage')
    
    return features

# 통계 출력
print("=" * 80)
print("심부전 주요 유형별 분포:")
print("=" * 80)
type_counts = hf_diagnoses_detailed['hf_type'].value_counts()
for hf_type, count in type_counts.items():
    pct = count / len(hf_diagnoses_detailed) * 100
    print(f"{hf_type:35} : {count:>6,}건 ({pct:>5.1f}%)")

# 가장 흔한 진단 코드별 상세 정보
print("\n" + "=" * 80)
print("가장 흔한 심부전 진단 코드 상세 (Top 10):")
print("=" * 80)
top_codes_detailed = hf_diagnoses_detailed.groupby(['icd_code', 'long_title', 'hf_type']).size().reset_index(name='count')
top_codes_detailed = top_codes_detailed.sort_values('count', ascending=False).head(10)

for idx, row in top_codes_detailed.iterrows():
    print(f"\n{row['icd_code']:10} | {row['count']:>5,}건 | {row['hf_type']}")
    print(f"           {row['long_title']}")
    
# 시각화 - 다차원 분류
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('심부전 유형별 다차원 분석', fontsize=18, fontweight='bold')

# 1. 주요 유형별 분포
ax1 = axes[0, 0]
type_counts_sorted = type_counts.sort_values(ascending=True)
colors_bar = plt.cm.Set3(range(len(type_counts_sorted)))
bars = ax1.barh(range(len(type_counts_sorted)), type_counts_sorted.values, 
                color=colors_bar, edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(type_counts_sorted)))
ax1.set_yticklabels(type_counts_sorted.index, fontsize=10)
ax1.set_xlabel('진단 건수', fontsize=11)
ax1.set_title('심부전 유형별 분포 (단일 분류)', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3, linestyle='--')

for i, val in enumerate(type_counts_sorted.values):
    ax1.text(val + 20, i, f'{val:,}', va='center', fontweight='bold', fontsize=9)

# 2. 시간 경과별 (Temporal)
ax2 = axes[0, 1]
temporal_mask = hf_diagnoses_detailed['hf_type'].str.contains('급성|만성|Acute|Chronic', na=False)
temporal_data = hf_diagnoses_detailed[temporal_mask]['hf_type'].value_counts()
if len(temporal_data) > 0:
    colors = ['#e74c3c', '#3498db', '#f39c12']
    wedges, texts, autotexts = ax2.pie(temporal_data.values,
                                         labels=temporal_data.index,
                                         autopct='%1.1f%%',
                                         colors=colors[:len(temporal_data)],
                                         startangle=90)
    ax2.set_title('시간 경과별 분류\n(Acute vs Chronic)', fontsize=13, fontweight='bold')
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')
else:
    ax2.text(0.5, 0.5, '데이터 없음', ha='center', va='center', fontsize=12)

# 3. 기능별 (Functional)
ax3 = axes[1, 0]
functional_mask = hf_diagnoses_detailed['hf_type'].str.contains('수축기|이완기|복합|Systolic|Diastolic|Combined', na=False)
functional_data = hf_diagnoses_detailed[functional_mask]['hf_type'].value_counts()
if len(functional_data) > 0:
    colors = ['#9b59b6', '#1abc9c', '#e67e22']
    wedges, texts, autotexts = ax3.pie(functional_data.values,
                                         labels=functional_data.index,
                                         autopct='%1.1f%%',
                                         colors=colors[:len(functional_data)],
                                         startangle=90)
    ax3.set_title('심장 기능별 분류\n(Systolic vs Diastolic)', fontsize=13, fontweight='bold')
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')
else:
    ax3.text(0.5, 0.5, '데이터 없음', ha='center', va='center', fontsize=12)

# 4. 해부학적 위치별 (Anatomical)
ax4 = axes[1, 1]
anatomical_mask = hf_diagnoses_detailed['hf_type'].str.contains('좌심실|우심실|양심실|Left|Right|Biventricular', na=False)
anatomical_data = hf_diagnoses_detailed[anatomical_mask]['hf_type'].value_counts()
if len(anatomical_data) > 0:
    colors = ['#2ecc71', '#e74c3c', '#f1c40f']
    wedges, texts, autotexts = ax4.pie(anatomical_data.values,
                                         labels=anatomical_data.index,
                                         autopct='%1.1f%%',
                                         colors=colors[:len(anatomical_data)],
                                         startangle=90)
    ax4.set_title('해부학적 위치별 분류\n(Left vs Right)', fontsize=13, fontweight='bold')
    for text in texts:
        text.set_fontsize(9)
    for autotext in autotexts:
        autotext.set_fontsize(10)
        autotext.set_fontweight('bold')
else:
    ax4.text(0.5, 0.5, '데이터 없음', ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()



**데이터 체크 및 전체 요약**

In [ ]:
print("전체 데이터 품질 체크\n")
print("=" * 80)

quality_check = {}

# 각 주요 테이블 체크
tables_to_check = {
    'patients': df_patients,
    'admissions': df_admissions,
    'diagnoses_icd': df_diagnoses_records
}

for table_name, df in tables_to_check.items():
    print(f"\n[{table_name.upper()}]")
    print("-" * 80)
    
    # 기본 정보
    print(f"  • 총 레코드 수: {len(df):,}")
    print(f"  • 컬럼 수: {len(df.columns)}")
    
    # 결측치
    total_missing = df.isnull().sum().sum()
    total_cells = df.shape[0] * df.shape[1]
    missing_pct = (total_missing / total_cells * 100)
    print(f"  • 전체 결측치: {total_missing:,} ({missing_pct:.2f}%)")
    
    # 중복
    if 'hadm_id' in df.columns:
        duplicates = df.duplicated(subset=['hadm_id']).sum()
        print(f"  • 중복 입원 ID: {duplicates:,}건")
    elif 'subject_id' in df.columns:
        duplicates = df.duplicated(subset=['subject_id']).sum()
        print(f"  • 중복 환자 ID: {duplicates:,}건")
    
    quality_check[table_name] = {
        'records': len(df),
        'columns': len(df.columns),
        'missing_pct': missing_pct
    }
